# Numbers quoted in the report

Reproduce the numbers of report section 3.3 from the cases and timings recorded
by [03_02](03_02_measure_latency.ipynb). The figures and the paired differences
between methods are in [03_03](03_03_compare_methods.ipynb).

In [1]:
import pandas as pd

## 1. Data

Times are in ms. Compute each time's median over repetitions, then join the
mask metadata.

In [2]:
METRICS = ["selection_ms", "io_ms", "wall_ms"]

In [3]:
cases   = pd.read_csv("cases.csv", float_precision="round_trip")
timings = pd.read_csv("timings.csv", float_precision="round_trip")

In [4]:
medians = timings.groupby("case_id")[METRICS].median()

results = cases.join(
    medians,
    on       = "case_id",
    validate = "one_to_one",
)

In [5]:
results = results.rename(columns={
    "tile_rows"    : "tile_width",
    "retained_pct" : "retained",
})

## 2. Traces and budgets

In [6]:
results.groupby(["n", "d"]).trace_id.nunique()

n     d   
896   128     32
      896     32
      4864    32
4864  896     32
Name: trace_id, dtype: int64

In [7]:
results.fraction.unique()

array([0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1. ])

## 3. Paper variants

Paper sorts its window scores on the GPU or on the CPU. Check that both
variants return the same mask for every trace and budget.

In [8]:
KEYS = ["trace_id", "fraction"]

gpu = results[results.method == "Paper greedy (GPU sort)"].set_index(KEYS)
cpu = results[results.method == "Paper greedy (CPU sort)"].set_index(KEYS)

In [9]:
gpu.intervals.eq(cpu.intervals).all()

np.True_

## 4. Medians at a 50% budget

For each shape and method, compute every column's median across traces.

In [10]:
FRACTION = 0.5
COLUMNS  = ["tile_width", "retained", "selection_ms", "io_ms", "wall_ms"]

In [11]:
selected = results[results.fraction == FRACTION]
summary  = selected.groupby(["n", "d", "method"])[COLUMNS].median()

### 4.1 $896\times896$

`Tiles (1s)` uses $B=\lceil s\rceil$ for the 240 KiB reference.

In [12]:
METHODS = [
    "Paper greedy (GPU sort)",
    "Paper greedy (CPU sort)",
    "Tiles (1s)",
    "Top-R",
]

In [13]:
summary.loc[(896, 896)].loc[METHODS].round(2)

,tile_width,retained,selection_ms,io_ms,wall_ms
method,,,,,
Paper greedy (GPU sort),NaN,61.36,1.04,0.38,1.49
Paper greedy (CPU sort),NaN,61.36,4.25,0.39,4.71
Tiles (1s),138.0,57.74,0.01,0.41,0.45
Top-R,1.0,74.02,0.03,1.99,2.05
